In [1]:
# ============================================================
# DreamSaver ML - Transaction Classifier Fine-Tuning
# ============================================================
# Run this notebook on Google Colab (free GPU runtime)
#
# STEP 1: Go to https://colab.research.google.com
# STEP 2: Click File → Upload Notebook → Upload this .ipynb or
#         paste this script into a new Colab notebook
# STEP 3: Go to Runtime → Change runtime type → GPU (T4)
# STEP 4: Run all cells
# ============================================================

# %% [Cell 1] Install required libraries
# !pip install transformers datasets torch accelerate huggingface_hub -q

# %% [Cell 2] Upload your dataset
# from google.colab import files
# uploaded = files.upload()  # Upload srilanka_transactions.csv

# %% [Cell 3] Load and prepare dataset
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_csv("srilanka_transactions.csv")
print(f"Total samples: {len(df)}")
print(f"Categories: {df['label'].nunique()}")
print(df['label'].value_counts())

# Create label mapping
labels = sorted(df['label'].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(labels)

print(f"\nLabel mapping: {label2id}")

# Convert labels to integers
df['label_id'] = df['label'].map(label2id)

# Split into train/test
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_dataset = Dataset.from_pandas(train_df[['description', 'label_id']].rename(columns={'label_id': 'label'}))
test_dataset = Dataset.from_pandas(test_df[['description', 'label_id']].rename(columns={'label_id': 'label'}))

dataset = DatasetDict({'train': train_dataset, 'test': test_dataset})
print(f"\nTrain: {len(train_dataset)}, Test: {len(test_dataset)}")

# %% [Cell 4] Load pretrained model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

print(f"Model loaded: {model_name}")
print(f"Number of categories: {num_labels}")

# %% [Cell 5] Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["description"],
        padding="max_length",
        truncation=True,
        max_length=64,  # Transaction descriptions are short
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Dataset tokenized!")

# %% [Cell 6] Set up training
from transformers import TrainingArguments, Trainer
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=15,           # More epochs for small dataset
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

# %% [Cell 7] Train!
print("Starting fine-tuning...")
trainer.train()

# %% [Cell 8] Evaluate
results = trainer.evaluate()
print(f"\n{'='*50}")
print(f"FINAL TEST ACCURACY: {results['eval_accuracy']:.2%}")
print(f"{'='*50}")

# %% [Cell 9] Test with real Sri Lankan transactions
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
)

test_transactions = [
    "KFC",
    "Keells Super",
    "Lanka IOC Fuel",
    "CEB Electricity",
    "Dialog Mobile",
    "Nawaloka Hospital",
    "Netflix",
    "Daraz.lk",
    "Monthly salary",
    "Uber Lanka",
    "Landlord Rent",
    "British Council",
    "AIA Insurance",
    "Bank loan EMI",
    "Temple donation",
]

print("\n--- Testing with Sri Lankan Transactions ---\n")
for tx in test_transactions:
    result = classifier(tx)[0]
    label = id2label[int(result['label'].split('_')[-1])] if 'LABEL' in result['label'] else result['label']
    print(f"  {tx:30s} → {label:15s} ({result['score']:.1%})")



Total samples: 218
Categories: 12
label
food             32
utilities        20
shopping         20
health           20
entertainment    20
education        20
transport        19
housing          15
insurance        15
income           15
loan             12
donation         10
Name: count, dtype: int64

Label mapping: {'donation': 0, 'education': 1, 'entertainment': 2, 'food': 3, 'health': 4, 'housing': 5, 'income': 6, 'insurance': 7, 'loan': 8, 'shopping': 9, 'transport': 10, 'utilities': 11}

Train: 174, Test: 44


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Training Loss,Validation Loss,Epoch,Accuracy
1.079772,1.464469,15,0.840909



FINAL TEST ACCURACY: 84.09%

--- Testing with Sri Lankan Transactions ---



[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  KFC                            → food            (43.1%)
  Keells Super                   → food            (37.2%)
  Lanka IOC Fuel                 → transport       (28.6%)
  CEB Electricity                → utilities       (29.8%)
  Dialog Mobile                  → utilities       (29.2%)
  Nawaloka Hospital              → health          (38.2%)
  Netflix                        → entertainment   (44.0%)
  Daraz.lk                       → shopping        (16.7%)
  Monthly salary                 → income          (36.6%)
  Uber Lanka                     → transport       (42.1%)
  Landlord Rent                  → income          (20.5%)
  British Council                → education       (20.6%)
  AIA Insurance                  → insurance       (31.7%)
  Bank loan EMI                  → loan            (28.8%)
  Temple donation                → donation        (19.6%)


HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a5475dd-3423347f1a5366520a205c43;75a7604c-3a1f-4f6e-a136-d9600c3b91cb)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [2]:
from huggingface_hub import notebook_login
notebook_login()


In [3]:

# %% [Cell 10] Push to Hugging Face Hub
# IMPORTANT: Replace 'YOUR_USERNAME' with your actual HF username
# First, login: run this in a cell:
#   from huggingface_hub import notebook_login
#   notebook_login()

HF_USERNAME = "samudu123"  # <-- CHANGE THIS
MODEL_NAME = "srilanka-transaction-classifier"

trainer.model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")

# Save label mapping as a JSON file
import json
label_config = {"label2id": label2id, "id2label": id2label, "labels": labels}
with open("label_config.json", "w") as f:
    json.dump(label_config, f, indent=2)

print(f"\n✅ Model pushed to: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")
print(f"✅ Label config saved to label_config.json")
print(f"\nYou can now use this model via the HF Inference API!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kto5ywv/model.safetensors:   0%|          |  575kB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]


✅ Model pushed to: https://huggingface.co/samudu123/srilanka-transaction-classifier
✅ Label config saved to label_config.json

You can now use this model via the HF Inference API!
